### Search Engine with Tools and Agents

In [1]:
## Arxiv--Reasearch 
##Tools creation

from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [2]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)
wiki.name

'wikipedia'

In [3]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv.name

'arxiv'

In [4]:
tools = [wiki,arxiv]

In [5]:
import os
import numpy as np
from dotenv import load_dotenv
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [6]:
## Custom tools
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\prath\Downloads\Langchain\Langchain_v1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [7]:
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
document = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap = 200).split_documents(docs)
vectordb = FAISS.from_documents(document,HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2'))

In [8]:
retriever = vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002381135B390>, search_kwargs={})

In [9]:
from langchain_core.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(retriever,"langsmith-search","Search any information about Langsmith")
retriever_tool.name

'langsmith-search'

In [10]:
tools = [wiki,arxiv,retriever_tool]

In [11]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\prath\\Downloads\\Langchain\\Langchain_v1\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='Search any information about Langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x00000238461CE480>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x00000238461CE520>)]

In [12]:
## Run all the tools with Agents and LLM models
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
api_key = os.getenv('GROQ_API_KEY')
from langchain_groq import ChatGroq
llm = ChatGroq(api_key=api_key,model='openai/gpt-oss-120b')

In [13]:
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023816C5C110>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002384653AD10>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [33]:
from langchain_classic import hub    ### New version
prompt = hub.pull("hwchase17/openai-functions-agent")

In [34]:
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [35]:
## Agents
from langchain_classic.agents import create_openai_tools_agent
agent = create_openai_tools_agent(llm,tools,prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

In [37]:
## Run the agent
from langchain_classic.agents import AgentExecutor

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [38]:
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

In [40]:
agent_executor.invoke({"input":"Tell me about Langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langsmith-search` with `{'query': 'LangSmith'}`


LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIToolsPolly AI assistantBetaCLISkillsAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agents and LLM applications.

It helps you trace requests, evaluate outputs, test prompts, and manage deployments in one place. LangSmith works with any agent stack, whether you’re using an existing framework or building your own. Prot

{'input': 'Tell me about Langsmith',
 'output': '**LangSmith – An Overview**\n\nLangSmith is a **framework‑agnostic observability and management platform** for building, debugging, testing, and deploying AI agents and LLM‑driven applications. It works with any stack—whether you’re using a popular framework (LangChain, CrewAI, Vercel AI SDK, etc.) or a completely custom implementation.\n\n---\n\n### Core Capabilities\n\n| Capability | What It Gives You | Why It Matters |\n|------------|-------------------|----------------|\n| **Observability** | End‑to‑end tracing of every request, prompt, model call, and tool use. Visual dashboards let you see the exact flow of an agent’s reasoning. | Faster debugging, pinpointing bottlenecks, and understanding failure modes. |\n| **Evaluation** | Built‑in metrics, custom evaluators, and versioned test suites to measure output quality over time. | Guarantees consistency, helps catch regressions, and provides data for continuous improvement. |\n| **Prom

In [42]:
agent_executor.invoke({"input":"What is Machine Learning"})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Machine learning'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks wit
Invoking: `wikipedia` with `{'query': 'Machine learning definition'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks wit**Machine learning (ML)** is a sub‑field of artificial intelligence that focuses on developing algorithms and statistical models that enable computers to **learn from data** and make predictions or decisions without being explicitly programmed for each specific task.

### Core Idea
- **Learning from examples**: An ML system

{'input': 'What is Machine Learning',
 'output': '**Machine learning (ML)** is a sub‑field of artificial intelligence that focuses on developing algorithms and statistical models that enable computers to **learn from data** and make predictions or decisions without being explicitly programmed for each specific task.\n\n### Core Idea\n- **Learning from examples**: An ML system is trained on a dataset containing input–output pairs (or just inputs) and automatically discovers patterns or relationships.\n- **Generalization**: After training, the model can apply what it has learned to new, unseen data, performing tasks such as classification, regression, clustering, or control.\n\n### How It Works (High‑level)\n1. **Data Collection** – Gather representative data for the problem.\n2. **Feature Representation** – Convert raw data into a form (features) the algorithm can process.\n3. **Model Selection** – Choose an algorithm (e.g., linear regression, decision tree, neural network) that defines

In [44]:
agent_executor.invoke({'input':"What is the paper 1706.03762 about?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`




c:\Users\prath\Downloads\Langchain\Langchain_v1\.venv\Lib\site-packages\langchain_community\utilities\arxiv.py:102: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  ).results()


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on c
Invoking: `arxiv` with `{'query': '1706.03762 abstract'}`




c:\Users\prath\Downloads\Langchain\Langchain_v1\.venv\Lib\site-packages\langchain_community\utilities\arxiv.py:105: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  ).results()


Published: 2006-12-27
Title: Generalizing the Paige-Tarjan Algorithm by Abstract Interpretation
Authors: Francesco Ranzato, Francesco Tapparo
Summary: The Paige and Tarjan algorithm (PT) for computing the coarsest refinement of a state partition whic
Invoking: `arxiv` with `{'query': 'arxiv 1706.03762 transformer'}`




...

Published: 2022-01-04
Title: PyramidTNT: Improved Transformer-in-Transformer Baselines with Pyramid Architecture
Authors: Kai Han, Jianyuan Guo, Yehui Tang, Yunhe Wang
Summary: Transformer networks have achieved great progress for computer vision tas
Invoking: `arxiv` with `{'query': 'arXiv:1706.03762'}`




c:\Users\prath\Downloads\Langchain\Langchain_v1\.venv\Lib\site-packages\langchain_community\utilities\arxiv.py:105: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  ).results()


Published: 2020-02-12
Title: GLU Variants Improve Transformer
Authors: Noam Shazeer
Summary: Gated Linear Units (arXiv:1612.08083) consist of the component-wise product of two linear projections, one of which is first passed through a sigmoid functio**Paper:** *Attention Is All You Need* (arXiv:1706.03762)  
**Authors:** Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin  
**Submitted:** June 2017 (published in *Advances in Neural Information Processing Systems* 2017)

---

### Core Idea  
The paper proposes a **new neural network architecture for sequence‑to‑sequence tasks that relies solely on attention mechanisms**, discarding the recurrent (RNN/LSTM/GRU) and convolutional components that dominated prior models. By doing so, the model—named the **Transformer**—can be trained much faster and more efficiently while achieving state‑of‑the‑art performance on machine translation benchmarks.

---

### Key Contributions 

{'input': 'What is the paper 1706.03762 about?',
 'output': '**Paper:** *Attention Is All You Need* (arXiv:1706.03762)  \n**Authors:** Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N.\u202fGomez, Łukasz Kaiser, Illia Polosukhin  \n**Submitted:** June\u202f2017 (published in *Advances in Neural Information Processing Systems* 2017)\n\n---\n\n### Core Idea  \nThe paper proposes a **new neural network architecture for sequence‑to‑sequence tasks that relies solely on attention mechanisms**, discarding the recurrent (RNN/LSTM/GRU) and convolutional components that dominated prior models. By doing so, the model—named the **Transformer**—can be trained much faster and more efficiently while achieving state‑of‑the‑art performance on machine translation benchmarks.\n\n---\n\n### Key Contributions  \n\n| Contribution | What It Is | Why It Matters |\n|--------------|------------|----------------|\n| **Self‑attention (scaled dot‑product attention)** | Computes pair

Failed to refresh cache entry rlm/rag-prompt: Connection error caused failure to GET /commits/rlm/rag-prompt/latest in LangSmith API. Please confirm your internet connection. ProxyError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/rlm/rag-prompt/latest (Caused by ProxyError(\'Unable to connect to proxy\', NewConnectionError("HTTPSConnection(host=\'172.16.69.78\', port=3128): Failed to establish a new connection: [WinError 10051] A socket operation was attempted to an unreachable network")))'))
Content-Length: None
API Key: lsv2_********************************************a6
Failed to refresh cache entry hwchase17/openai-functions-agent: Connection error caused failure to GET /commits/hwchase17/openai-functions-agent/latest in LangSmith API. Please confirm your internet connection. ProxyError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hw